In [ ]:
# Cell 1: Setup and Imports
import sys
import numpy as np
from skimage import io, color

def load_image_as_numpy(file_path):
    # Load image
    img = io.imread(file_path)
    # Convert to grayscale if it is RGB
    if img.ndim == 3:
        img = color.rgb2gray(img)
    # Resize to 28x28 if necessary
    img = resize(img, (28, 28))
    # Reshape to (1, 28, 28)
    return img.reshape(1, 1, 28, 28)

# Add source directory to the path
sys.path.append('./src')

from module import Module, Linear, Conv2d, MaxPool2d, Flatten, ReLU, Sigmoid, Dropout, CrossEntropyLoss, SGD
from functions import F

ModuleNotFoundError: No module named 'torchvision'

In [ ]:
# Cell 2: Define the MNIST CNN Architecture
class MNIST_CNN(Module):
    def __init__(self):
        super().__init__()
        # 28x28x1 Input -> Conv Layer (8 filters)
        self.conv1 = Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1)
        self.relu1 = ReLU()
        self.pool1 = MaxPool2d(kernel_size=2, stride=2)
        
        # Flatten: 8 channels * 14 * 14
        self.flatten = Flatten()
        
        # Fully Connected Layer: 1568 inputs -> 10 output classes
        self.fc1 = Linear(in_features=(8 * 14 * 14), out_features=10)

    def forward(self, x):
        # Pass data through the layers
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)
        x = self.flatten(x)
        x = self.fc1(x)
        return F.Softmax(x)

In [ ]:
import numpy as np
from sklearn.datasets import fetch_openml

print("Loading MNIST data...")
# Fetch data
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')

# take 100 for training and the next 20 for testing
train_data = mnist.data[:100]
test_data = mnist.data[100:120]

train_labels_raw = mnist.target[:100].astype(int)
test_labels_raw = mnist.target[100:120].astype(int)

# Normalize and reshape (Batch, Channel, Height, Width)
train_images = train_data.reshape(-1, 1, 28, 28) / 255.0
test_images = test_data.reshape(-1, 1, 28, 28) / 255.0

# Convert labels to One-Hot
train_labels = np.eye(10)[train_labels_raw]
test_labels = np.eye(10)[test_labels_raw]

print(f"Training set shape: {train_images.shape}")
print(f"Test set shape: {test_images.shape}")

In [ ]:
# Cell 3.5: Visualize
import matplotlib.pyplot as plt

# Display 5 random images
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
indices = np.random.choice(len(train_images), 5, replace=False)

for idx, ax in enumerate(axes):
    # Reshape back to 28x28 for plotting
    ax.imshow(train_images[indices[idx]].reshape(28, 28), cmap='gray')
    ax.set_title(f"Label: {np.argmax(train_labels[indices[idx]])}")
    ax.axis('off')
plt.show()

In [ ]:
# Cell 4: Training Loop
model = MNIST_CNN()
criterion = CrossEntropyLoss()
optimizer = SGD(layers=[model.conv1, model.fc1], lr=0.01)

num_samples = len(train_images)
epochs = 5

for epoch in range(epochs):
    epoch_loss = 0
    correct = 0
    
    for i in range(num_samples):
        # 1. Forward Pass
        prediction = model(train_images[i])
        
        # 2. Loss Calculation
        loss = criterion.forward(prediction, train_labels[i])
        epoch_loss += loss
        
        # Check accuracy
        if np.argmax(prediction) == np.argmax(train_labels[i]):
            correct += 1
            
        # 3. Backward Pass
        d_out = criterion.backward()
        d_out = model.fc1.backward(d_out)
        d_out = model.flatten.backward(d_out)
        d_out = model.pool1.backward(d_out)
        d_out = model.relu1.backward(d_out)
        d_out = model.conv1.backward(d_out)
        
        # 4. Update Weights
        optimizer.step()
        optimizer.zero_grad()

    print(f"Epoch {epoch+1} | Loss: {epoch_loss/num_samples:.4f} | Accuracy: {(correct/num_samples)*100:.2f}%")

print("Training Complete!")

In [ ]:
# Cell 5: Error Analysis on Test Data
wrong_predictions = []

print("\n--- Running Error Analysis on Test Set ---")
for i in range(len(test_images)):
    prediction = model(test_images[i]) # Use the test images
    target = test_labels[i]
    
    pred_idx = np.argmax(prediction)
    true_idx = np.argmax(target)
    
    # We define "high confidence error" as > 50% probability for the wrong class
    if pred_idx != true_idx and prediction[pred_idx] > 0.5:
        wrong_predictions.append((i, pred_idx, true_idx, prediction[pred_idx]))
        print(f"Index {i}: Predicted {pred_idx}, Actual {true_idx} (Confidence: {prediction[pred_idx]:.2f})")

print(f"\nTotal high-confidence errors found: {len(wrong_predictions)}")